In [ ]:
from __future__ import annotations
import polars as pl
import re
import math
from collections import Counter, defaultdict
import marimo as mo

In [ ]:
train_df = pl.read_parquet("data/train.parquet")

In [ ]:
bench_df = pl.read_parquet("data/benchmark_queries.parquet")

In [ ]:
queries_df = pl.concat([
    train_df.select(["search_query", "search_infm_params_text"]),
    bench_df.select(["search_query", "search_infm_params_text"])
])

### Ключи фильтров

**Что я сразу понял**

По `train` увидел, что можно попробовать достать ключи фильтров из полей. Пробовал по `items`, но в итоге пришел к мнению, что там не имеет смысла их брать, для моей задачи хватит достать только те ключи, которые есть в запросах.

**Как решил задачу**

Самым эффективным способом оказалось вручную поставить базовые ключи и смотреть покрытие, добавляя необходимые ключи. Я какое-то время пытался перебрать вообще все ключи, которые присутствуют в запросах, но пришел к выводу, что это не имеет смысла, так как там могут быть сложные "вложенности" признаков, которые мне ничего особо не дают.

**Что еще пробовал**

N-граммный поиск по частоте, было многовато мусора, который невозможно было очистить. Пытался определять по энтропии, тоже в общем не дало никакой пользы.

**В итоге**

Получил вяглядящий неплохо список ключей, по которому дальше распарсил их значения.

In [ ]:
KEYS_LOWER = [
    "вид услуги",
    "тип услуги",
    "тип товара",
    "вид товара",
    "срочная услуга (мультистатус)",
    "кто оказывает услуги",
    "рейтинг пользователя",
    "тип объявления",
    "сфера деятельности",
    "график работы, дни недели",
    "марка",
    "состояние",
    "вид техники",
    "открытие в сегменте авито для бизнеса",
    "марка авто",
    "график работы v2",
    "график работы"
]

KEYS = [s[0].upper() + s[1:] for s in KEYS_LOWER]


def build_key_regex(keys: list[str]) -> re.Pattern:
    keys_sorted = sorted(set(keys), key=len, reverse=True)
    pattern = "|".join(re.escape(k) for k in keys_sorted)
    return re.compile(rf"\b({pattern})")


def coverage_report(query_texts: pl.Series, keys: list[str]) -> dict:
    key_re = build_key_regex(keys)
    texts = [t for t in query_texts.drop_nulls().unique().to_list() if t]
    uncovered = [t for t in texts if not key_re.search(t) if "Слова в описании" not in t]
    return {
        "total": len(texts),
        "uncovered": len(uncovered),
        "ratio": len(uncovered) / max(len(texts), 1),
        "examples": uncovered[:10],
    }

In [ ]:
coverage_report(queries_df["search_infm_params_text"], KEYS)

### Парсинг значений ключей

In [ ]:
def _is_capitalized(word: str) -> bool:
    """Слово, у которого первая буква (кириллица/латиница) — заглавная."""
    for ch in word:
        if ch.isalpha():
            return ch.isupper()
    return False


def _is_boundary_token(word: str) -> bool:
    """
    Обрывает ли токен значение.

    Не обрывают:
    - строчные слова (продолжение значения);
    - аббревиатуры из заглавных букв (ТО, BMW, ИП, ГБО, МКПП, ХВС);
    - токены, начинающиеся с не-буквы ((LADA), '1,5', 'т.').

    Обрывают:
    - слова с заглавной первой буквой, не аббревиатуры (Вид, Моторное, Услуги);
    - служебные маркеры в квадратных/фигурных скобках ([поиск], {...}).
    """
    if not word:
        return False
    first = word[0]

    # маркеры [поиск], {...} — граница
    if first in "[{":
        return True

    # не-заглавная первая буква — не граница
    if not first.isupper():
        return False

    # аббревиатуры: все буквы заглавные, длина <= 4
    letters = [c for c in word if c.isalpha()]
    if letters and all(c.isupper() for c in letters) and len(letters) <= 4:
        return False

    return True


def _find_value_bounds(segment: str) -> tuple[int, int]:
    """Начало — первый токен после ключа. Конец — перед первым boundary-токеном."""
    if not segment.strip():
        return 0, 0

    tokens = [(m.start(), m.end(), m.group()) for m in re.finditer(r"[^\s]+", segment)]
    if not tokens:
        return 0, len(segment)

    start = tokens[0][0]
    for i in range(1, len(tokens)):
        if _is_boundary_token(tokens[i][2]):
            return start, tokens[i][0]
    return start, len(segment)


def parse_params(text: str, key_regex: re.Pattern) -> dict[str, list[str]]:
    """
    key-value парсер с обрезкой значения по границе заглавных слов.

    text     — сырое значение поля *_infm_params_text (без .lower()!)
    key_regex — case-sensitive regex, собранный из title-case ключей
    """
    if not text:
        return {}

    matches = list(key_regex.finditer(text))
    if not matches:
        return {}

    result: dict[str, list[str]] = {}
    for i, m in enumerate(matches):
        key = m.group(1)
        seg_start = m.end()
        seg_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        segment = text[seg_start:seg_end]

        v_start, v_end = _find_value_bounds(segment)
        value = segment[v_start:v_end].strip(" ,;.")

        result.setdefault(key, [])
        if value:
            result[key].append(value)
    return result

In [ ]:
items_df = pl.read_parquet("data/benchmark_items.parquet")

In [ ]:
key_re = build_key_regex(KEYS)

queries_parsed = queries_df.with_columns(
    pl.col("search_infm_params_text")
      .fill_null("")
      .map_elements(lambda t: parse_params(t, key_re), return_dtype=pl.Object)
      .alias("q_params")
)

items_parsed = items_df.with_columns(
    pl.col("item_infm_params_text")
      .fill_null("")
      .map_elements(lambda t: parse_params(t, key_re), return_dtype=pl.Object)
      .alias("i_params")
)

In [ ]:
def examples_per_key(
    df: pl.DataFrame,
    col: str = "q_params",
    raw_col: str = "search_infm_params_text",
    n: int = 5,
) -> None:
    """Просто выводит примеры вытащенных фильтров"""
    buckets: dict[str, list[str]] = {}
    for raw, d in zip(df[raw_col].to_list(), df[col].to_list()):
        if not d or not raw:
            continue
        for k, vals in d.items():
            buckets.setdefault(k, [])
            if len(buckets[k]) < n:
                buckets[k].append(f"{raw[:100]} → {k} = {vals}")

    for k in sorted(buckets, key=lambda x: -len(buckets[x])):
        print(f"\n=== {k} ({len(buckets[k])} shown) ===")
        for ex in buckets[k]:
            print("  ", ex)

In [ ]:
examples_per_key(queries_parsed)


=== Рейтинг пользователя (5 shown) ===
   Рейтинг пользователя 4 звезды и выше → Рейтинг пользователя = ['4 звезды и выше']
   Рейтинг пользователя 4 звезды и выше Вид услуги Деловые услуги Тип услуги → Рейтинг пользователя = ['4 звезды и выше']
   Рейтинг пользователя 4 звезды и выше → Рейтинг пользователя = ['4 звезды и выше']
   Рейтинг пользователя 4 звезды и выше Кто оказывает услуги Частный исполнитель Вид услуги Монтаж и ус → Рейтинг пользователя = ['4 звезды и выше']
   Рейтинг пользователя 4 звезды и выше Где вы оказываете услуги У себя дома Где вы оказываете услуги В → Рейтинг пользователя = ['4 звезды и выше']

=== Тип услуги (5 shown) ===
   Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье → Тип услуги = ['СПА-услуги, массаж']
   Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка → Тип услуги = ['Ремонт квартир и домов под ключ']
   Тип услуги Сборка и ремонт мебели Вид услуги Ремонт и отделка → Тип услуги = ['Сборка и ремонт меб

In [ ]:
examples_per_key(items_parsed, col="i_params", raw_col="item_infm_params_text")


=== Вид услуги (5 shown) ===
   Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная це → Вид услуги = ['Компьютерная помощь']
   Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, л → Вид услуги = ['Обучение, курсы']
   Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип усл → Вид услуги = ['Красота, здоровье']
   Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу Раб → Вид услуги = ['Строительство']
   Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ленина → Вид услуги = ['Обучение, курсы']

=== График работы (5 shown) ===
   Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная це → График работы = ['от 34200', 'до 82800']
   Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское